In [1]:
class BytePairEncoder:
  def __init__(self, max_tokens: int, corpus: str):
      self.max_tokens = max_tokens
      corpus_bytes = bytearray(corpus, encoding="ascii")
      self.token_pairs = {}
      while 128 + len(self.token_pairs) < self.max_tokens:
          print(f"\r{round((128 + len(self.token_pairs))/self.max_tokens*100.0, 3)}%", end="")
          frequencies = {}
          for i in range(len(corpus_bytes) - 1):
              pair = (corpus_bytes[i], corpus_bytes[i + 1])
              if pair in frequencies.keys():
                  frequencies[pair] += 1
              else:
                  frequencies[pair] = 1
          frequencies = dict(sorted(frequencies.items(), key=lambda x: x[1]))
          new_pair = list(frequencies.keys())[-1]
          self.token_pairs[new_pair] = 128 + len(self.token_pairs)
          new_corpus_bytes = []
          skip_next = False
          for j in range(len(corpus_bytes) - 1):
              pair = (corpus_bytes[j], corpus_bytes[j + 1])
              if not skip_next:
                  if pair == new_pair:
                      new_corpus_bytes.append(self.token_pairs[pair])
                      skip_next = True
                  else:
                      new_corpus_bytes.append(corpus_bytes[j])
              else:
                  skip_next = False
          if not skip_next:
              new_corpus_bytes.append(corpus_bytes[-1])
          corpus_bytes = new_corpus_bytes

  @property
  def reversed_token_pairs(self):
      return {v: k for k, v in self.token_pairs.items()}

  def encode(self, corpus):
      corpus_bytes = list(bytearray(corpus, encoding="ascii"))
      for check_pair in self.token_pairs.keys():
        new_corpus_bytes = []
        skip_next = False
        for i in range(len(corpus_bytes) - 1):
          if not skip_next:
            pair = (corpus_bytes[i], corpus_bytes[i + 1])
            if pair == check_pair:
              new_corpus_bytes.append(self.token_pairs[pair])
              skip_next = True
            else:
              new_corpus_bytes.append(corpus_bytes[i])
          else:
            skip_next = False
        if not skip_next:
          new_corpus_bytes.append(corpus_bytes[-1])
        corpus_bytes = new_corpus_bytes
      return corpus_bytes


  def decode(self, tokens: list[int]):
      corpus_bytes = tokens[:]
      while not all(map(lambda x: x < 128, corpus_bytes)):
          new_corpus_bytes = []
          for byte in corpus_bytes:
              if byte in self.reversed_token_pairs.keys():
                  pair = self.reversed_token_pairs[byte]
                  new_corpus_bytes.append(pair[0])
                  new_corpus_bytes.append(pair[1])
              else:
                  new_corpus_bytes.append(byte)
          corpus_bytes = new_corpus_bytes
      return "".join([i.to_bytes().decode("ascii") for i in corpus_bytes])

In [2]:
import torch
from torch.utils.data import Dataset

class BPEDataset(Dataset):
  def __init__(self, encoder: BytePairEncoder, text: str, context: int):
    self.corpus = encoder.encode(text)
    self.encoder = encoder
    self.context = context
    self.corpus = torch.tensor(self.corpus, dtype=torch.long)
  def __len__(self):
    return len(self.corpus)-self.context
  def __getitem__(self, idx):
    return self.corpus[idx:idx+self.context], self.corpus[idx+1:idx+self.context+1]

In [ ]:
import torch
import torch.nn as nn
from math import sin, cos

class LLM(nn.Module):
  def __init__(self, num_tokens: int, d_model: int, max_tokens: int, attention_layers: int, attention_heads: int, ff_hiddens: int, ff_hidden_size: int):
    super().__init__()
    self.max_tokens = max_tokens
    self.num_tokens = num_tokens
    self.d_model = d_model
    self.attention_layers = attention_layers
    self.attention_heads = attention_heads
    self.ff_hiddens = ff_hiddens
    self.ff_hidden_size = ff_hidden_size
    self.embedding = nn.Embedding(self.num_tokens, self.d_model)

    attn_mask = ~torch.tril(torch.full((self.max_tokens, self.max_tokens), True))
    self.register_buffer("attn_mask", attn_mask)

    positional_encodings = torch.zeros((self.max_tokens, self.d_model))
    for pos in range(max_tokens):
      positional_encodings[pos, 0::2] = torch.sin(pos / (10000 ** (torch.arange(0, d_model, 2) / d_model)))
      positional_encodings[pos, 1::2] = torch.cos(pos / (10000 ** (torch.arange(0, d_model, 2) / d_model)))
    self.register_buffer("positional_encodings", positional_encodings)
    self.multiheadattentions = nn.ModuleList(list([nn.MultiheadAttention(self.d_model, self.attention_heads, batch_first=True) for _ in range(self.attention_layers)]))
    self.feed_forwards = nn.ModuleList()
    for _ in range(self.attention_layers):
      feed_forward = nn.Sequential()
      feed_forward.append(nn.Linear(self.d_model, self.ff_hidden_size))
      feed_forward.append(nn.ReLU())
      for _ in range(self.ff_hiddens):
        feed_forward.append(nn.Linear(self.ff_hidden_size, self.ff_hidden_size))
        feed_forward.append(nn.ReLU())
      feed_forward.append(nn.Linear(self.ff_hidden_size, self.d_model))
      self.feed_forwards.append(feed_forward)
    self.layer_norms1 = nn.ModuleList()
    self.layer_norms2 = nn.ModuleList()

    for i in range(self.attention_layers):
      self.layer_norms1.append(nn.LayerNorm(self.d_model))
      self.layer_norms2.append(nn.LayerNorm(self.d_model))

  def forward(self, x, mask=True):
    tokens = x
    embedded = self.embedding(tokens)
    pos = self.positional_encodings[:embedded.shape[1]]
    embedded = embedded + pos.unsqueeze(0)
    for i in range(self.attention_layers):
      residual = embedded
      embedded = self.layer_norms1[i](embedded)
      embedded = residual + self.multiheadattentions[i](embedded, embedded, embedded, need_weights=False, attn_mask = self.attn_mask[:embedded.shape[1], :embedded.shape[1]] if mask else None)[0]
      residual = embedded
      embedded = self.layer_norms2[i](embedded)
      embedded = residual + self.feed_forwards[i](embedded)
    logits = torch.matmul(embedded, self.embedding.weight.T)
    return logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as functional
import pickle
from torch.utils.data import DataLoader
import json
import os
import time
from encoder import BytePairEncoder
from dataset import BPEDataset
from llm import LLM

with open("config.json", "r") as f:
  params = json.load(f)

with open ("shakespeare.txt", "r") as f:
  shakespeare = f.read()

if not os.path.exists(params["encoder_path"]):
  encoder = BytePairEncoder(params["vocab_size"], shakespeare)
  with open(params["encoder_path"], "wb") as f:
    pickle.dump(encoder, f)
else:
  with open(params["encoder_path"], "rb") as f:
    encoder = pickle.load(f)

train_dataset = BPEDataset(encoder,
                          shakespeare,
                          params["context_length"])
batch_per_epoch = len(train_dataset)//params["batch_size"]
train_loader = DataLoader(train_dataset,
                          batch_size=params["batch_size"],
                          shuffle=True)

model = LLM(num_tokens=128+len(encoder.token_pairs),
            d_model=params["d_model"],
            max_tokens=params["context_length"],
            attention_layers=params["attn_layers"],
            attention_heads=params["attn_heads"],
            ff_hiddens=params["feedforward_layers"],
            ff_hidden_size=params["feedforward_layer_size"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on {str(device).upper()}")
model.to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"])
for epoch in range(params["epochs"]):
  for i, data in enumerate(train_loader):
    start = time.perf_counter_ns()
    x, y = data
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    outputs = model(x)
    loss = loss_func(outputs.transpose(-2, -1), y)
    loss.backward()
    optimizer.step()
    print(f"Loss at epoch {epoch+1}/{params["epochs"]}, batch {i}/{batch_per_epoch}: {loss.item()}, time: {round((time.perf_counter_ns()-start)*10**-9, 4)} secs")
  with open(params["model_path"], "wb") as f:
    torch.save(model, f)

Running on CPU


In [ ]:
import torch, pickle
import json

with open("encoder.pkl", "rb") as f:
  encoder = pickle.load(f)

with open("model.pkl", "rb") as f:
  old_state = torch.load(f, weights_only=False).state_dict()

with open("config.json", "rb") as f:
  params = json.load(f)

model = LLM(num_tokens=128+len(encoder.token_pairs),
            d_model=params["d_model"],
            max_tokens=params["context_length"],
            attention_layers=params["attn_layers"],
            attention_heads=params["attn_heads"],
            ff_hiddens=params["feedforward_layers"],
            ff_hidden_size=params["feedforward_layer_size"])

model.load_state_dict(old_state)

model.to("cpu")
text = "I"
tokens = torch.tensor(encoder.encode(text))
with torch.no_grad():
  while len(tokens) < 1000:
    x = torch.unsqueeze(tokens[-32:], dim=0)
    logits = model(x, mask=True)[0, -1]
    probs = torch.softmax(logits, dim=-1)
    choice = torch.unsqueeze(torch.multinomial(probs, num_samples=1)[-1], dim=0)
    new_token = choice.tolist()
    tokens = torch.cat((tokens, torch.tensor(new_token)), dim=-1)
  print(encoder.decode(tokens.tolist()))